<a href="https://colab.research.google.com/github/byter66/Aegis/blob/main/monuments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, io, re, time, shutil, json
import numpy as np, pandas as pd, requests
from PIL import Image

Mounted at /content/drive


In [3]:
!wget -q -nc https://s3.amazonaws.com/google-landmark/metadata/train.csv
!wget -q -nc https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv

train = pd.read_csv("train.csv", usecols=["id", "url", "landmark_id"])
cats = pd.read_csv("train_label_to_category.csv")
cats["name"] = cats["category"].str.extract(r"Category:(.+)$")[0].fillna("").str.replace("_", " ")
kw = ("Taj Mahal|Red Fort|India Gate|Gateway of India|Hawa Mahal|Charminar|Qutub|Qutb|"
      "Victoria Memorial|Mysore Palace|Golden Temple|Lotus Temple|Howrah Bridge|Konark|Jodhpur"
      "Meenakshi|Hampi|Sanchi|Jantar Mantar|Humayun|Fatehpur Sikri|Agra Fort|Amber|Rashtrapati")
cand = cats[cats["name"].str.contains(kw, case=False, regex=True)].copy()
cand["images"] = cand["landmark_id"].map(train["landmark_id"].value_counts())
cand = cand.sort_values("images", ascending=False)
print(cand[["landmark_id", "name", "images"]].head(40).to_string())

        landmark_id                                                         name  images
10419         10419                          Qutb Minar and its monuments, Delhi     945
64792         64792                                   Victoria Memorial, Kolkata     600
107801       107801                                       Lahori Gate (Red Fort)     484
105496       105496                                                     Red Fort     439
172056       172056                                                       Amberd     424
183170       183170                                                    Agra Fort     423
65068         65068                                        Jantar Mantar (Delhi)     399
60730         60730                                             Gateway of India     342
124675       124675                                                   India Gate     331
160240       160240                                            Taj Mahal, Bhopal     296
55780         55780  

In [4]:
chosen = cand[cand.images >= 60].head(15)
# remove unwanted rows by hand if needed, e.g. chosen = chosen.drop(index=[123, 456])
print(chosen[["name", "images"]])

UA = {"User-Agent": "IIITK-PrivacyProject/1.0 (student project; contact: nivedithagireesan@gmail.com)"}

def fetch(url):
    for attempt in range(4):
        r = requests.get(url, headers=UA, timeout=20)
        if r.status_code == 200 and r.headers.get("Content-Type", "").startswith("image"):
            return r.content
        if r.status_code in (429, 503):
            time.sleep(5 * (attempt + 1)); continue
        print("status", r.status_code, r.text[:100]); return None
    return None

for u in train[train.landmark_id == chosen.iloc[0].landmark_id].url.head(5):
    print("OK" if fetch(u) else "FAILED", u)
    time.sleep(1)

                                       name  images
10419   Qutb Minar and its monuments, Delhi     945
64792            Victoria Memorial, Kolkata     600
107801               Lahori Gate (Red Fort)     484
105496                             Red Fort     439
172056                               Amberd     424
183170                            Agra Fort     423
65068                 Jantar Mantar (Delhi)     399
60730                      Gateway of India     342
124675                           India Gate     331
160240                    Taj Mahal, Bhopal     296
55780             Victoria Memorial, London     288
179405           Sint-Lambertuskerk (Buren)     281
51329                            Amber Fort     267
2838                           Lotus Temple     230
50663                    Rashtrapati Bhavan     186
OK https://upload.wikimedia.org/wikipedia/commons/3/37/Qutab_Minar%2C_Delhi%2C_India_%2821206051545%29.jpg
OK https://upload.wikimedia.org/wikipedia/commons/2/2c/Qutub_

In [7]:
out = "/content/landmark_sorted"
for _, row in chosen.iterrows():
    cls = re.sub(r"[^A-Za-z0-9_ -]", "", row["name"]).strip().replace(" ", "_")[:50]
    d = os.path.join(out, cls); os.makedirs(d, exist_ok=True)
    got = len(os.listdir(d))
    urls = train[train.landmark_id == row.landmark_id].sample(frac=1, random_state=0)
    for _, r in urls.iterrows():
        if got >= 80: break
        dest = os.path.join(d, r["id"] + ".jpg")
        if os.path.exists(dest): continue
        data = fetch(r["url"])
        if data:
            try:
                im = Image.open(io.BytesIO(data)).convert("RGB")
                im.thumbnail((400, 400)); im.save(dest, quality=90); got += 1
            except Exception:
                pass
        time.sleep(0.7)
    print(cls, got)

Qutb_Minar_and_its_monuments_Delhi 80
Victoria_Memorial_Kolkata 80
Lahori_Gate_Red_Fort 80
Red_Fort 80
Amberd 80
Agra_Fort 80
Jantar_Mantar_Delhi 80
Gateway_of_India 80
India_Gate 80
Taj_Mahal_Bhopal 80
Victoria_Memorial_London 80
Sint-Lambertuskerk_Buren 80
Amber_Fort 80
Lotus_Temple 80
Rashtrapati_Bhavan 80


In [8]:
shutil.make_archive("/content/drive/MyDrive/landmark_sorted", "zip", "/content/landmark_sorted")
print("zipped")

zipped


In [3]:
!unzip -q "/content/drive/MyDrive/landmark_sorted.zip" -d /content/landmark_sorted

replace /content/landmark_sorted/Amber_Fort/654ebf17cf8420c5.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/landmark_sorted/Amber_Fort/aa7ef2036dcceadf.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/landmark_sorted/Amber_Fort/4963f5749f7232c6.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/landmark_sorted/Amber_Fort/07deb700be6ceacf.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/landmark_sorted/Amber_Fort/347ef99121bcbc96.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/landmark_sorted/Amber_Fort/854f5bdfcdc48510.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/landmark_sorted/Amber_Fort/a4597b40b01dbdc2.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/landmark_sorted/Amber_Fort/dcb405eaf0236ad9.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/landmark_sorted/Amber_Fort/cb44a8f2718848ef.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/landmark_sorted/Amber_Fort/43

In [4]:
root = "/content/landmark_sorted"; bad_dir = "/content/bad_images"; os.makedirs(bad_dir, exist_ok=True)
n_bad = 0
for dp, _, fs in os.walk(root):
       for f in fs:
           p = os.path.join(dp, f)
           try:
               with Image.open(p) as im: im.load()
           except Exception:
               shutil.move(p, os.path.join(bad_dir, f)); n_bad += 1
print(n_bad, "bad files moved")

0 bad files moved


In [6]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

datagen = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)
train_data = datagen.flow_from_directory(root, target_size=(224,224), batch_size=16, class_mode='categorical', subset='training')
val_data = datagen.flow_from_directory(root, target_size=(224,224), batch_size=16, class_mode='categorical', subset='validation')
print(train_data.num_classes, list(train_data.class_indices))

Found 960 images belonging to 15 classes.
Found 240 images belonging to 15 classes.
15 ['Agra_Fort', 'Amber_Fort', 'Amberd', 'Gateway_of_India', 'India_Gate', 'Jantar_Mantar_Delhi', 'Lahori_Gate_Red_Fort', 'Lotus_Temple', 'Qutb_Minar_and_its_monuments_Delhi', 'Rashtrapati_Bhavan', 'Red_Fort', 'Sint-Lambertuskerk_Buren', 'Taj_Mahal_Bhopal', 'Victoria_Memorial_Kolkata', 'Victoria_Memorial_London']


In [7]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False
x = GlobalAveragePooling2D()(base.output)
x = Dense(128, activation='relu')(x)
out = Dense(train_data.num_classes, activation='softmax')(x)
model = Model(base.input, out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, validation_data=val_data, epochs=2)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/2
60/60 ━━━━━━━━━━━━━━━━━━━━ 22s 157ms/step - accuracy: 0.5250 - loss: 1.6438 - val_accuracy: 0.6500 - val_loss: 1.1244
Epoch 2/2
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 76ms/step - accuracy: 0.8417 - loss: 0.5404 - val_accuracy: 0.7542 - val_loss: 0.8597


In [8]:
history = model.fit(train_data, validation_data=val_data, epochs=10)
print("val acc:", history.history["val_accuracy"][-1])
os.makedirs("/content/drive/MyDrive/landmark_module/models", exist_ok=True)
model.save("/content/drive/MyDrive/landmark_module/models/landmark_model.keras")
json.dump(list(train_data.class_indices.keys()), open("/content/drive/MyDrive/landmark_module/models/class_names.json", "w"))

Epoch 1/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - accuracy: 0.9365 - loss: 0.2569 - val_accuracy: 0.7417 - val_loss: 0.8460
Epoch 2/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - accuracy: 0.9781 - loss: 0.1384 - val_accuracy: 0.7375 - val_loss: 0.8457
Epoch 3/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.9906 - loss: 0.0837 - val_accuracy: 0.7292 - val_loss: 0.8424
Epoch 4/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - accuracy: 0.9979 - loss: 0.0497 - val_accuracy: 0.7750 - val_loss: 0.8095
Epoch 5/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - accuracy: 0.9979 - loss: 0.0302 - val_accuracy: 0.7542 - val_loss: 0.8211
Epoch 6/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 1.0000 - loss: 0.0185 - val_accuracy: 0.7500 - val_loss: 0.7965
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 1.0000 - loss: 0.0138 - val_accuracy: 0.7542 - val_loss: 0.8137
Epoch 8/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - accuracy: 1.0000 - loss: 0.0107 - val_accuracy: 0.7542 - v